In [1]:
import pandas as pd
import numpy as np

def reporte_calidad(df, nombre_dataset):
    """
    Genera un reporte de calidad de datos similar a Sweetviz.
    Úsalo con CUALQUIER DataFrame para diagnóstico rápido.
    """
    print("=" * 70)
    print(f"DIAGNÓSTICO DE CALIDAD — {nombre_dataset.upper()}")
    print("=" * 70)
    
    # 1. Vista general
    print(f"\n📊 VISTA GENERAL")
    print(f"   Filas: {df.shape[0]:,}  |  Columnas: {df.shape[1]}")
    print(f"   Memoria: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
    
    # 2. Análisis por columna
    print(f"\n📋 ANÁLISIS POR COLUMNA")
    print(f"{'Columna':<25} {'Tipo':<10} {'Únicos':>8} {'Nulos':>8} {'Nulos%':>7} {'Ceros':>8} {'Ceros%':>7}")
    print("-" * 78)
    
    alertas = []
    
    for col in df.columns:
        tipo = str(df[col].dtype)[:9]
        unicos = df[col].nunique()
        nulos = df[col].isnull().sum()
        nulos_pct = nulos / len(df) * 100
        
        if df[col].dtype in ['int64', 'float64']:
            ceros = (df[col] == 0).sum()
            ceros_pct = ceros / len(df) * 100
        else:
            ceros = 0
            ceros_pct = 0
        
        print(f"{col:<25} {tipo:<10} {unicos:>8,} {nulos:>8,} {nulos_pct:>6.1f}% {ceros:>8,} {ceros_pct:>6.1f}%")
        
        # Detectar alertas
        if nulos_pct > 50:
            alertas.append(f"🔴 ALTA: {col} tiene {nulos_pct:.0f}% nulos — considerar eliminar")
        elif nulos_pct > 10:
            alertas.append(f"🟡 MEDIA: {col} tiene {nulos_pct:.0f}% nulos — investigar")
        if ceros_pct > 20:
            alertas.append(f"🟡 MEDIA: {col} tiene {ceros_pct:.0f}% ceros — ¿son reales?")
        if unicos == 1:
            alertas.append(f"🔴 ALTA: {col} tiene 1 solo valor — columna inútil, eliminar")
    
    # 3. Estadísticas numéricas
    numericas = df.select_dtypes(include=[np.number]).columns
    if len(numericas) > 0:
        print(f"\n📈 ESTADÍSTICAS NUMÉRICAS")
        stats = df[numericas].describe().T
        stats['skew'] = df[numericas].skew()
        stats['kurt'] = df[numericas].kurt()
        print(stats[['mean', '50%', 'std', 'min', 'max', 'skew', 'kurt']].to_string())
    
    # 4. Correlaciones fuertes
    if len(numericas) > 1:
        print(f"\n🔗 CORRELACIONES FUERTES (|r| > 0.7)")
        corr = df[numericas].corr()
        pares = []
        for i in range(len(corr.columns)):
            for j in range(i+1, len(corr.columns)):
                r = corr.iloc[i, j]
                if abs(r) > 0.7:
                    pares.append((corr.columns[i], corr.columns[j], r))
        if pares:
            for c1, c2, r in sorted(pares, key=lambda x: abs(x[2]), reverse=True):
                print(f"   {c1} ↔ {c2}: {r:.3f}")
        else:
            print("   Ninguna correlación > 0.7")
    
    # 5. Alertas
    print(f"\n⚠️ ALERTAS DE CALIDAD ({len(alertas)} encontradas)")
    if alertas:
        for a in alertas:
            print(f"   {a}")
    else:
        print("   ✅ Sin alertas — dataset en buen estado")
    
    print("\n" + "=" * 70)
    return alertas

In [2]:
# LEONALI
df_leonali = pd.read_parquet(r'C:\Users\Said\Documents\Datasets\Alimentos\Dataset1_anonimizado.parquet')
alertas_leonali = reporte_calidad(df_leonali, "Leonali (Ensaladas)")

DIAGNÓSTICO DE CALIDAD — LEONALI (ENSALADAS)

📊 VISTA GENERAL
   Filas: 581,306  |  Columnas: 15
   Memoria: 117.0 MB

📋 ANÁLISIS POR COLUMNA
Columna                   Tipo         Únicos    Nulos  Nulos%    Ceros  Ceros%
------------------------------------------------------------------------------
IdCliente                 str             130        0    0.0%        0    0.0%
IdProducto                str             233        0    0.0%        0    0.0%
FechaProceso              datetime6     3,608        0    0.0%        0    0.0%
Producto                  str             214        0    0.0%        0    0.0%
Categoria1                str              11        0    0.0%        0    0.0%
Categoria2                str               9        0    0.0%        0    0.0%
marca                     str               6        0    0.0%        0    0.0%
Canal1                    str               7        0    0.0%        0    0.0%
Canal2                    str              11        0    0

In [3]:
# FARMACIAS
df_farmacias = pd.read_parquet(r'C:\Users\Said\Documents\Datasets\Farmacias\farmacias_ventas_det_anonimizado.parquet')
alertas_farmacias = reporte_calidad(df_farmacias, "Farmacias")

DIAGNÓSTICO DE CALIDAD — FARMACIAS

📊 VISTA GENERAL
   Filas: 18,320,551  |  Columnas: 12
   Memoria: 2436.6 MB

📋 ANÁLISIS POR COLUMNA
Columna                   Tipo         Únicos    Nulos  Nulos%    Ceros  Ceros%
------------------------------------------------------------------------------
ID_MARCA                  int64             2        0    0.0%        0    0.0%
ID_TIENDA                 int64           147        0    0.0%        0    0.0%
folio_venta               str        12,511,522        0    0.0%        0    0.0%
tipo_transaccion          int64             2        0    0.0% 15,739,805   85.9%
SKU                       int64        12,654        0    0.0%        2    0.0%
cantidad                  float64         426        0    0.0%       15    0.0%
precio_unit               float64      88,869        0    0.0%      942    0.0%
iva_unit                  float64       8,777        0    0.0% 11,361,682   62.0%
fecha_venta               str             701        0    0

In [4]:
# RESTAURANTE
df_restaurante = pd.read_csv(r'C:\Users\Said\Documents\Datasets\SoftRes\cheqdet.csv')
alertas_restaurante = reporte_calidad(df_restaurante, "Restaurante (cheqdet)")

DIAGNÓSTICO DE CALIDAD — RESTAURANTE (CHEQDET)

📊 VISTA GENERAL
   Filas: 398,208  |  Columnas: 17
   Memoria: 64.1 MB

📋 ANÁLISIS POR COLUMNA
Columna                   Tipo         Únicos    Nulos  Nulos%    Ceros  Ceros%
------------------------------------------------------------------------------
foliodet                  int64        61,650        0    0.0%        0    0.0%
movimiento                int64           202        0    0.0%        0    0.0%
comanda                   float64           1  398,207  100.0%        0    0.0%
cantidad                  float64          46        0    0.0%        9    0.0%
idproducto                int64           556        0    0.0%        0    0.0%
descuento                 float64          27        0    0.0%  384,896   96.7%
precio                    float64         477        0    0.0%  106,183   26.7%
impuesto1                 int64             2        0    0.0%  396,281   99.5%
impuesto2                 int64             1        0    